In [2]:
import numpy as np

from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split


In [3]:
# ============================================================
# Load Data
# ============================================================
X = np.loadtxt("waves.csv", delimiter=",")
Y = np.loadtxt("enzymes.csv", delimiter=",")

if Y.ndim == 1:
    Y = Y.reshape(-1, 1)
n_targets = Y.shape[1]
n_features = X.shape[1]

In [4]:
n_samples, n_features = X.shape
n_targets = Y.shape[1]
print(f"X shape: {X.shape}, Y shape: {Y.shape}")

# Single consistent train/test split used everywhere
test_size = 0.2
random_state = 42

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=test_size, random_state=random_state
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train shape: {Y_train.shape}, Test shape: {Y_test.shape}")


X shape: (258, 1868), Y shape: (258, 5)
Train shape: (206, 1868), Test shape: (52, 1868)
Train shape: (206, 5), Test shape: (52, 5)


In [5]:
def evaluate_model(
    X, y,
    model_name,
    ridge_alpha=1.0,
    en_alpha=1.0,
    en_l1_ratio=0.5,
    lasso_alpha=1.0,
    max_iter=30000,
    max_pls_components=15,
    cv_splits=5,
    verbose=True,
    random_state=42
):
    """
    Train and evaluate model on (X, y).
    Scaling rules:
        - PLS, Ridge → StandardScaler
        - ElasticNet, Lasso → NO scaling
    PLS finds best n_components via CV.
    """

    rng = np.random.RandomState(random_state)

    # ============
    #     PLS
    # ============
    if model_name.lower() == "pls":
        max_comp = min(max_pls_components, X.shape[1], X.shape[0] - 1)
        best_score = -np.inf
        best_k = 1

        for k in range(1, max_comp + 1):
            pls = PLSRegression(n_components=k, scale=True)
            
            cv = KFold(n_splits=cv_splits, shuffle=True,
                    random_state=rng.randint(0, 1_000_000))
            
            score = cross_val_score(pls, X, y, cv=cv, scoring="r2").mean()
            
            if score > best_score:
                best_score = score
                best_k = k

        # Train final model
        model = PLSRegression(n_components=best_k, scale=True)
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(best n_components = {best_k})"

    # ============
    #     RIDGE
    # ============
    elif model_name.lower() == "ridge":
        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=ridge_alpha)
        )
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(alpha={ridge_alpha})"

    # ============
    #  ELASTIC NET
    # ============
    elif model_name.lower() == "elasticnet":
        model = ElasticNet(alpha=en_alpha, l1_ratio=en_l1_ratio, max_iter=max_iter)
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(alpha={en_alpha}, l1={en_l1_ratio})"

    # ============
    #     LASSO
    # ============
    elif model_name.lower() == "lasso":
        model = Lasso(alpha=lasso_alpha, max_iter=max_iter)
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(alpha={lasso_alpha})"

    else:
        raise ValueError("Invalid model_name")

    # ============
    #   METRICS
    # ============
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    rep = 100 * rmse / np.mean(np.abs(y))

    if verbose:
        print(f"  {model_name.upper()} {extra}")
        print(f"    R²   = {r2:.4f}")
        print(f"    RMSE = {rmse:.4f}")
        print(f"    REP% = {rep:.2f}")

    return model, {"r2": r2, "rmse": rmse, "rep": rep}


In [14]:
def pso_feature_selection(
    X,
    y,
    n_particles=40,
    n_iterations=30,
    inertia=0.7,
    cognitive=1.5,
    social=1.5,
    random_state=42,
    verbose=True
):
    rng = np.random.RandomState(random_state)
    n_samples, n_features = X.shape

    # -------------------------
    # Fitness function (same logic you used)
    # -------------------------
    def fitness(particle):
        cols = np.where(particle == 1)[0]
        if len(cols) == 0:
            return -9999

        pls = PLSRegression(n_components=8)
        pls.fit(X[:, cols], y)
        pred = pls.predict(X[:, cols]).ravel()
        return r2_score(y, pred)

    # -------------------------
    # Initialize swarm
    # -------------------------
    positions = (rng.rand(n_particles, n_features) < 0.1).astype(float)  # float positions
    velocities = rng.uniform(-1, 1, size=(n_particles, n_features))

    # Avoid empty particles
    for i in range(n_particles):
        if not positions[i].any():
            positions[i, rng.randint(0, n_features)] = 1.0

    personal_best_pos = positions.copy()
    personal_best_scores = np.array([fitness(p) for p in positions])

    global_best_idx = np.argmax(personal_best_scores)
    global_best_pos = personal_best_pos[global_best_idx].copy()
    global_best_score = personal_best_scores[global_best_idx]

    if verbose:
        print(f"Initial global best R² = {global_best_score:.4f}, "
              f"features = {global_best_pos.sum()}")

    # -------------------------
    # PSO main loop
    # -------------------------
    for it in range(1, n_iterations + 1):
        for i in range(n_particles):
            r1 = rng.rand(n_features)
            r2 = rng.rand(n_features)

            velocities[i] = (
                inertia * velocities[i] + cognitive * r1 * (personal_best_pos[i] - positions[i]) + social * r2 * (global_best_pos - positions[i])
            )

            # Sigmoid for binary PSO
            prob = 1 / (1 + np.exp(-velocities[i]))
            positions[i] = (rng.rand(n_features) < prob).astype(float)

            # Avoid empty particle
            if not positions[i].any():
                positions[i, rng.randint(0, n_features)] = 1.0

            score = fitness(positions[i])

            # Update personal best
            if score > personal_best_scores[i]:
                personal_best_scores[i] = score
                personal_best_pos[i] = positions[i].copy()

                # Update global best
                if score > global_best_score:
                    global_best_score = score
                    global_best_pos = positions[i].copy()

        if verbose:
            print(f"Iteration {it:02d}: "
                  f"best R² = {global_best_score:.4f}, "
                  f"features = {global_best_pos.sum()}")

    # -------------------------
    # Final selected features
    # -------------------------
    selected_indices = np.where(global_best_pos == 1)[0]
    X_sel = X[:, selected_indices]

    if verbose:
        print("\nFinal evaluation on training data with selected features "
              f"({len(selected_indices)} features):")

    # -------------------------
    # Train & evaluate models (UNCHANGED)
    # -------------------------
    model_candidates = ['pls', 'ridge', 'elasticnet', 'lasso']
    models_array = []

    best_model = None
    best_model_name = None
    best_metrics = None
    best_r2 = -np.inf

    for name in model_candidates:
        model, metrics = evaluate_model(
            X_sel, y,
            model_name=name,
            verbose=verbose,
            random_state=random_state
        )
        models_array.append(model)

        if metrics["r2"] > best_r2:
            best_r2 = metrics["r2"]
            best_model = model
            best_model_name = name
            best_metrics = metrics

    if verbose:
        print(f"\nBest model on training: {best_model_name.upper()} "
              f"with R² = {best_r2:.4f}")

    return selected_indices, best_model, best_model_name, models_array, best_metrics

In [7]:
from sklearn.cross_decomposition import PLSRegression

def is_pls(model):
    if hasattr(model, "steps"):
        return isinstance(model.steps[-1][1], PLSRegression)
    return False
from sklearn.linear_model import Ridge

def is_ridge(model):
    if hasattr(model, "steps"):
        return isinstance(model.steps[-1][1], Ridge)
    return False
from sklearn.linear_model import ElasticNet, Lasso

def is_elasticnet(model):
    return isinstance(model, ElasticNet)

def is_lasso(model):
    return isinstance(model, Lasso)


In [8]:
from sklearn.metrics import r2_score, mean_squared_error

def evaluate_on_validation(model, X_val_sel, y_val, verbose=True):
    """
    Evaluate an ALREADY TRAINED model on validation data.
    No retraining, no scaling here.
    Scaling and hyperparameters are already baked into `model`.
    """
    model.fit(X_val_sel, y_val)
    y_pred = model.predict(X_val_sel).ravel()

    r2 = r2_score(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mean_y = np.mean(np.abs(y_val))
    rep = 100.0 * rmse / mean_y if mean_y != 0 else np.nan

    if verbose:
        print(f"    Validation R²   = {r2:.4f}")
        print(f"    Validation RMSE = {rmse:.4f}")
        print(f"    Validation REP% = {rep:.2f}")

    return {"r2": r2, "rmse": rmse, "rep": rep}


In [9]:
def get_model_name(model):
    # Pipeline → model.steps list
    if hasattr(model, "steps"):
        final_step = model.steps[-1][1]
        return final_step.__class__.__name__
    else:
        return model.__class__.__name__


In [16]:
# Example: X_train, Y_train, X_val, Y_val already defined.
# Y_* shape: (n_samples, 5)

n_enzymes = Y_train.shape[1]

for enzyme_idx in range(n_enzymes):
    print("=" * 70)
    print(f"Enzyme {enzyme_idx + 1}")
    
    y_train = Y_train[:, enzyme_idx]
    y_val = Y_test[:, enzyme_idx]
    
    # Run GA feature selection for this enzyme
    selected_features, trained_model, model_name, models_array, train_metrics = pso_feature_selection(
        X_train,
        y_train,
        n_particles=30,
        n_iterations=20,
        random_state=42 + enzyme_idx,
        verbose=True
    )

    
    print(f"\nEnzyme {enzyme_idx + 1}:")
    print(f"  Selected {len(selected_features)} features.")
    print(f"  Best training model: {model_name.upper()}")
    
    # Evaluate on validation data using same selected features
    X_val_sel = X_test[:, selected_features]
    print("  Validation performance for ALL models:")

    for model in models_array:
        model_name = get_model_name(model)
        print(f"\n  Model: {model_name}")

        val_metrics = evaluate_on_validation(
            model,
            X_val_sel=X_test[:, selected_features],
            y_val=y_val
        )



Enzyme 1
Initial global best R² = 0.7072, features = 204.0
Iteration 01: best R² = 0.7964, features = 1084.0
Iteration 02: best R² = 0.7964, features = 1084.0
Iteration 03: best R² = 0.7976, features = 1032.0
Iteration 04: best R² = 0.7991, features = 992.0
Iteration 05: best R² = 0.8010, features = 997.0
Iteration 06: best R² = 0.8024, features = 964.0
Iteration 07: best R² = 0.8024, features = 964.0
Iteration 08: best R² = 0.8087, features = 945.0
Iteration 09: best R² = 0.8087, features = 945.0
Iteration 10: best R² = 0.8087, features = 945.0
Iteration 11: best R² = 0.8087, features = 945.0
Iteration 12: best R² = 0.8087, features = 945.0
Iteration 13: best R² = 0.8087, features = 945.0
Iteration 14: best R² = 0.8087, features = 945.0
Iteration 15: best R² = 0.8087, features = 945.0
Iteration 16: best R² = 0.8087, features = 945.0
Iteration 17: best R² = 0.8087, features = 945.0
Iteration 18: best R² = 0.8087, features = 945.0
Iteration 19: best R² = 0.8087, features = 945.0
Iterati

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.875e+05, tolerance: 1.823e+05
  model = cd_fast.enet_coordinate_descent(


  LASSO (alpha=1.0)
    R²   = 0.9712
    RMSE = 505.1907
    REP% = 22.50

Best model on training: RIDGE with R² = 0.9982

Enzyme 2:
  Selected 964 features.
  Best training model: RIDGE
  Validation performance for ALL models:

  Model: PLSRegression
    Validation R²   = 0.8572
    Validation RMSE = 1108.9629
    Validation REP% = 60.08

  Model: Ridge
    Validation R²   = 0.9999
    Validation RMSE = 23.3109
    Validation REP% = 1.26

  Model: ElasticNet
    Validation R²   = 0.5717
    Validation RMSE = 1920.7121
    Validation REP% = 104.07

  Model: Lasso
    Validation R²   = 0.9997
    Validation RMSE = 52.8974
    Validation REP% = 2.87
Enzyme 3
Initial global best R² = 0.6914, features = 203.0
Iteration 01: best R² = 0.8090, features = 1076.0
Iteration 02: best R² = 0.8090, features = 1076.0
Iteration 03: best R² = 0.8090, features = 1076.0
Iteration 04: best R² = 0.8178, features = 953.0
Iteration 05: best R² = 0.8178, features = 953.0
Iteration 06: best R² = 0.8178, feat

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.748e+05, tolerance: 2.849e+04
  model = cd_fast.enet_coordinate_descent(


  LASSO (alpha=1.0)
    R²   = 0.9303
    RMSE = 310.5643
    REP% = 56.75

Best model on training: RIDGE with R² = 0.9996

Enzyme 3:
  Selected 951 features.
  Best training model: RIDGE
  Validation performance for ALL models:

  Model: PLSRegression
    Validation R²   = 0.9636
    Validation RMSE = 342.4701
    Validation REP% = 34.57

  Model: Ridge
    Validation R²   = 1.0000
    Validation RMSE = 10.6576
    Validation REP% = 1.08

  Model: ElasticNet
    Validation R²   = 0.4763
    Validation RMSE = 1298.6577
    Validation REP% = 131.08

  Model: Lasso
    Validation R²   = 0.9990
    Validation RMSE = 56.5835
    Validation REP% = 5.71
Enzyme 4
Initial global best R² = 0.7079, features = 183.0
Iteration 01: best R² = 0.8067, features = 1060.0
Iteration 02: best R² = 0.8067, features = 1060.0
Iteration 03: best R² = 0.8111, features = 1007.0
Iteration 04: best R² = 0.8120, features = 959.0
Iteration 05: best R² = 0.8120, features = 959.0
Iteration 06: best R² = 0.8120, featu